In [ ]:
import torch
from datasets import load_dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import numpy as np
import librosa
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"

# model_name = 'whisper_large_v3_mansi'
# model_name = 'whisper_medium_mansi'
# model_name = 'whisper_small_mansi_v1'
# model_name = 'whisper_small_mansi'
model_name = 'whisper_small_mansi_synth_v2'

stt_model_path = f"../stt/models/{model_name}"

In [ ]:
tts_path = "../data/tts_dataset"
tts_test_dataset = load_dataset('csv', data_files=tts_path+'/test/metadata.csv', delimiter='\t', split='train')

stt_path = "../data/stt_dataset"
stt_test_dataset = load_dataset('csv', data_files=stt_path+'/test/metadata.csv', delimiter='\t', split='train')

bonus_path = "../data/bonus_dataset"
bonus_test_dataset = load_dataset('csv', data_files=bonus_path+'/test/metadata.csv', delimiter='\t', split='train')

In [ ]:
import evaluate
import re
import unicodedata

def normalize(text: str) -> str:
    text = text.lower()
    
    normalized_chars = []
    for ch in text:
        cat = unicodedata.category(ch)
        if cat.startswith("L") or cat == "Zs":
            normalized_chars.append(ch)
        elif unicodedata.combining(ch) and ch in "̄":
            normalized_chars.append(ch)
        elif ch in ["ӈ"]:
            normalized_chars.append(ch)

    return re.sub(r"\s+", " ", "".join(normalized_chars)).strip()

metric_wer = evaluate.load("wer")
metric_cer = evaluate.load("cer")


def compute_metrics(pred_text, test_text):
    orh_wer = 100 * metric_wer.compute(predictions=pred_text, references=test_text)
    orh_cer = 100 * metric_cer.compute(predictions=pred_text, references=test_text)

    pred_str_norm = [normalize(pred) for pred in pred_text]
    label_str_norm = [normalize(label) for label in test_text]

    pred_str_norm = [
        pred_str_norm[i] for i in range(len(pred_str_norm)) if len(label_str_norm[i]) > 0
    ]
    label_str_norm = [ 
        label_str_norm[i] for i in range(len(label_str_norm))if len(label_str_norm[i]) > 0
    ]

    wer = 100 * metric_wer.compute(predictions=pred_str_norm, references=label_str_norm)
    cer = 100 * metric_cer.compute(predictions=pred_str_norm, references=label_str_norm)

    return {"wer": wer, "cer": cer, "orh_wer": orh_wer, "orh_cer": orh_cer}

In [ ]:
wer_list = []
cer_list = []
orh_wer_list = []
orh_cer_list = []

processor = WhisperProcessor.from_pretrained(stt_model_path, language="hungarian", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(stt_model_path).to(device)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None
model.generation_config._from_model_config = True

dataset = pd.DataFrame(columns=['model_out', 'label', 'wer', 'cer', 'orh_wer', 'orh_cer'])

for sample in tts_test_dataset:
    pred_file, sr = librosa.load(f'{tts_path}/test/wavs/{sample['audio_file']}', sr=16000)
    test_text = sample['text']

    inputs = processor(pred_file, sampling_rate=16000, return_tensors="pt").to(device)
    pred_ids = model.generate(inputs.input_features, max_length=225)
    pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]

    metrics = compute_metrics([pred_text], [test_text])

    wer_list.append(metrics['wer'])
    cer_list.append(metrics['cer'])
    orh_wer_list.append(metrics['orh_wer'])
    orh_cer_list.append(metrics['orh_cer'])
    new_row = pd.DataFrame([{
        'model_out': pred_text, 
        'label': test_text, 
        'wer': metrics['wer'], 
        'cer': metrics['cer'], 
        'orh_wer': metrics['orh_wer'], 
        'orh_cer': metrics['orh_cer']}])
    dataset = pd.concat([dataset, new_row], ignore_index=True)


for sample in stt_test_dataset:
    pred_file, sr = librosa.load(f'{stt_path}/test/wavs/{sample['audio_file']}', sr=16000)
    test_text = sample['text']

    inputs = processor(pred_file, sampling_rate=16000, return_tensors="pt").to(device)
    pred_ids = model.generate(inputs.input_features, max_length=225)
    pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]

    metrics = compute_metrics([pred_text], [test_text])
    new_row = pd.DataFrame([{
        'model_out': pred_text, 
        'label': test_text, 
        'wer': metrics['wer'], 
        'cer': metrics['cer'], 
        'orh_wer': metrics['orh_wer'], 
        'orh_cer': metrics['orh_cer']}])
    dataset = pd.concat([dataset, new_row], ignore_index=True)


for sample in bonus_test_dataset:
    pred_file, sr = librosa.load(f'{bonus_path}/test/wavs/{sample['audio_file']}', sr=16000)
    test_text = sample['text']

    inputs = processor(pred_file, sampling_rate=16000, return_tensors="pt").to(device)
    pred_ids = model.generate(inputs.input_features, max_length=225)
    pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]

    metrics = compute_metrics([pred_text], [test_text])
    new_row = pd.DataFrame([{
        'model_out': pred_text, 
        'label': test_text, 
        'wer': metrics['wer'], 
        'cer': metrics['cer'], 
        'orh_wer': metrics['orh_wer'], 
        'orh_cer': metrics['orh_cer']}])
    dataset = pd.concat([dataset, new_row], ignore_index=True)

dataset.to_csv(f'predicts_{model_name}.csv', index=False, sep="\t")